In [12]:
import pandas as pd
from IPython.display import display

print('--- STEP 1: LOAD & UNDERSTAND THE DATA ---')

# Direct CSV export link
#url = "https://docs.google.com/spreadsheets/d/106rVYbCbHIljj3CKQtQnBp5rDmGtZA4ecyAEgQ277Dg/export?format=csv&gid=796927918"
file_path = "../data/raw/antimicrobial_raw.csv"
print("Loading dataset...\n")
df = pd.read_csv(file_path)

display(df.head())

print("\n")
print("---  DATASET SHAPE ---")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}\n")

print("--- DATASET INFO ---")
df.info()
print("\n")



--- STEP 1: LOAD & UNDERSTAND THE DATA ---
Loading dataset...



,DOI,YEAR,MATERIAL,PRECURSOR,Synthesis_Method,Size_nm,Plant Extract Type,More about Plant Extract Type,Bacterial Strain,Inhibition Zone (mm),MIC(ug/ml),MBC,ROS Generation,Gram Status,Resistant Strain,Resistant Type,IZ_Censored
0,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,NaN,NaN,S. aureus,20.0,0.125,NaN,Yes,Gram-positive,No,NaN,NaN
1,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,NaN,NaN,Other,20.0,0.5,NaN,Yes,Gram-positive,No,NaN,NaN
2,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,NaN,NaN,Other,20.0,0.5,NaN,Yes,Gram-positive,No,NaN,NaN
3,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,NaN,NaN,Other,20.0,0.5,NaN,Yes,Gram-positive,No,NaN,NaN
4,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,NaN,NaN,Other,20.0,0.25,NaN,Yes,Gram-positive,Yes,MRSA,NaN




---  DATASET SHAPE ---
Total Rows: 211
Total Columns: 17

--- DATASET INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   DOI                            206 non-null    str    
 1   YEAR                           211 non-null    int64  
 2   MATERIAL                       211 non-null    str    
 3   PRECURSOR                      211 non-null    str    
 4   Synthesis_Method               211 non-null    str    
 5   Size_nm                        136 non-null    float64
 6   Plant Extract Type             94 non-null     str    
 7   More about Plant Extract Type  105 non-null    str    
 8   Bacterial Strain               211 non-null    str    
 9   Inhibition Zone (mm)           129 non-null    float64
 10  MIC(ug/ml)                     169 non-null    str    
 11  MBC                            77 non-nu

In [13]:
import numpy as np

print('--- STEP 2: DATA CLEANING & HANDLING MISSING VALUES ---')

# Display missing values percentage before cleaning
print('Percentage of missing values per column (before cleaning):')
print((df.isnull().sum() / len(df) * 100).round(2))
print('\n')

# Clean column names by stripping whitespace
df.columns = df.columns.str.strip()

# List of columns to fill with 'Unknown' for categorical data
categorical_cols_to_fill = [
    'DOI', # Added DOI as per user request to keep and fill missing
    'Plant Extract Type',
    'ROS Generation',
    'Resistant Type',
    'IZ_Censored'
]

# List of columns to process for numeric conversion (including ranges) and median imputation
numeric_cols_to_process = [
    'Size_nm',
    'MIC(ug/ml)',
    'MBC',
    'Concentration (µg/mL)'
]

# Helper function to parse numeric values, handle ranges, or return NaN
def parse_numeric_or_range(value):
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except ValueError:
        if '-' in s:
            try:
                parts = [float(p.strip()) for p in s.split('-')]
                return sum(parts) / len(parts)
            except ValueError:
                return np.nan
        else:
            return np.nan

# Handle categorical columns: fill with 'Unknown'
for col in categorical_cols_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Handle numeric columns: convert to numeric (including range parsing), then fill NaNs with the median
for col in numeric_cols_to_process:
    if col in df.columns:
        # Apply the custom parsing function
        df[col] = df[col].apply(parse_numeric_or_range)
        # Ensure the column is numeric, coercing any remaining errors
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].isnull().any(): # Only fill if there are NaNs after conversion/parsing
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)

# Drop 'Resistant Strain' column
df = df.drop(columns=['Resistant Strain' , 'More about Plant Extract Type'], errors='ignore')

# Display info and head of the cleaned DataFrame
print('--- Cleaned DATASET INFO ---')
df.info()
print('\n')
display(df.head())

--- STEP 2: DATA CLEANING & HANDLING MISSING VALUES ---
Percentage of missing values per column (before cleaning):
DOI                               2.37
YEAR                              0.00
MATERIAL                          0.00
PRECURSOR                         0.00
Synthesis_Method                  0.00
Size_nm                          35.55
Plant Extract Type               55.45
More about Plant Extract Type    50.24
Bacterial Strain                  0.00
Inhibition Zone (mm)             38.86
MIC(ug/ml)                       19.91
MBC                              63.51
ROS Generation                   45.02
Gram Status                       0.00
Resistant Strain                 16.11
Resistant Type                   80.57
IZ_Censored                      59.24
dtype: float64


--- Cleaned DATASET INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                ----

,DOI,YEAR,MATERIAL,PRECURSOR,Synthesis_Method,Size_nm,Plant Extract Type,Bacterial Strain,Inhibition Zone (mm),MIC(ug/ml),MBC,ROS Generation,Gram Status,Resistant Type,IZ_Censored
0,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,S. aureus,20.0,0.125,130.0,Yes,Gram-positive,Unknown,Unknown
1,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,Other,20.0,0.500,130.0,Yes,Gram-positive,Unknown,Unknown
2,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,Other,20.0,0.500,130.0,Yes,Gram-positive,Unknown,Unknown
3,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,Other,20.0,0.500,130.0,Yes,Gram-positive,Unknown,Unknown
4,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,Other,20.0,0.250,130.0,Yes,Gram-positive,MRSA,Unknown


In [14]:
df.to_csv('antimicrobial_clean.csv', index=False)
print("Cleaned dataset saved.")

Cleaned dataset saved.
